#### Bronze vs Silver Testing – Transactions Table

This document outlines the data quality and reconciliation tests performed to validate the ingestion of the `transactions` table from Bronze to Silver.

The goal of these tests is to ensure:
- No data loss
- No duplicate business keys
- No invalid records in Silver
- Monetary values remain consistent
- Pipeline is idempotent and reliable



#### 🧾 Tables Under Test

- Bronze: `coffee.bronze.transactions`
- Silver: `coffee.silver.transactions`
- Quarantine: `coffee.silver.transactions_quarantine`

In [0]:
-- Test 1: Basic row count comparison
-- Silver count should be <= Bronze count
-- Difference is expected due to filtering/quarantine

SELECT 'bronze' AS layer, COUNT(*) AS record_count
FROM coffee.bronze.transactions

UNION ALL

SELECT 'silver' AS layer, COUNT(*) AS record_count
FROM coffee.silver.transactions;


In [0]:
-- Test 2: Mandatory columns must NOT be NULL in Silver
-- Any non-zero count indicates a data quality failure

SELECT COUNT(*) AS invalid_silver_records
FROM coffee.silver.transactions
WHERE
  transaction_id IS NULL
  OR created_at IS NULL
  OR original_amount IS NULL
  OR final_amount IS NULL;


In [0]:
-- Test 3: Each transaction_id must be unique in Silver
-- Ensures deduplication logic is working

SELECT transaction_id, COUNT(*) AS cnt
FROM coffee.silver.transactions
GROUP BY transaction_id
HAVING COUNT(*) > 1;


In [0]:
-- Test 4: Silver should not contain records that never existed in Bronze
-- Ensures no phantom records are created

SELECT transaction_id
FROM coffee.silver.transactions

EXCEPT

SELECT transaction_id
FROM coffee.bronze.transactions;


In [0]:
-- Test 5: All valid Bronze records should either be in Silver or Quarantine
-- This identifies valid records accidentally dropped

SELECT transaction_id
FROM coffee.bronze.transactions
WHERE
  transaction_id IS NOT NULL
  AND created_at IS NOT NULL
  AND original_amount IS NOT NULL
  AND final_amount IS NOT NULL

EXCEPT

SELECT transaction_id
FROM coffee.silver.transactions;


In [0]:
-- Test 6: Sum reconciliation of amount columns
-- Silver sums should be <= Bronze sums
-- Difference represents filtered / quarantined records

SELECT 'bronze' AS layer,
       SUM(original_amount) AS sum_original_amount,
       SUM(final_amount)    AS sum_final_amount
       from
       coffee.bronze.transactions

UNION ALL

SELECT 'silver' AS layer,
       SUM(original_amount) AS sum_original_amount,
       SUM(final_amount)    AS sum_final_amount
FROM coffee.silver.transactions;
